---
title : "Analyse van Tijdseries"
execute:
    eval: false
    output: asis
    error: false
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

Hier laten we in iets meer detail zien hoe we een tijdseries analyse doen

In [1]:
# zorg dat alle variable goed staan
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

In [2]:
from talk_with_wiski.utils.logging_utils import setup_logging, logger
from pathlib import Path

logging_path = setup_logging(log_level="WARN", log_dir=str(Path.cwd()))
# bijvoobeeld @Andries:
logger.info(f"Logging path: {logging_path}")

In [3]:
# to retrieve a ts_id belonging to a specific station_no and parameter_type
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput
from talk_with_wiski.tools import get_timeseries_list

# to do analysis on timeseries
from talk_with_wiski.models import (
    AnalyseTimeseriesStatsInput,
    AnalyseTimeseriesThresholdsInput,
)
from talk_with_wiski.tools import (
    analyse_timeseries_statistics,
    analyse_timeseries_trend,
    analyse_timeseries_thresholds,
)

# INPUTS
# vind een station met waterhoogte-data
partypenm = "WNS8487"  # dit is waterhoogte
station_no = "1198_WTH_BOV"  # dit is een station_no voor maken interessant figuur
station_name = "Bovenstrooms"  # dit is de naam van het station

wlev_thresholds_dict = {
    # station_no: threshold_value
    "1198_WTH_BOV": 5.3,
    "7127_WTH_BEN": -0.5,
    "7132_WTH_UIT": 0.5,
}

# maken van input object
get_timeseries_list_input = GetTimeseriesListInput(
    parametertype_name=partypenm,
    station_no=station_no,
    station_name=station_name,
)
# uitlezen van timeseries beschikbaar voor dit station
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

data = get_timeseries_list_output.model_dump()
print(data)

# lijst van de beschikbare ts_id's
for item in data["data"]:
    print(
        f"TS ID: {item['ts_id']} | station_name: {item['station_name']} | station_no: {item['station_no']} | ts_name: {item['ts_name']}"
    )

# select from get_timeseries_list_output the ts_id belonging to station_no and parameter_type for station_name
# and data type (ts_name) "ts_name": "Uur.Gem",
ts_id_in = None
for item in data["data"]:
    if item["ts_name"] == "Uur.Gem":
        ts_id_in = str(item["ts_id"])
        break

print(f"using ts_id: {ts_id_in}")

# for statistics and trend, same object can be used
analyse_timeseries_input = AnalyseTimeseriesStatsInput(
    ts_id=ts_id_in,  # belongs to stat_no "1198_WTH_BOV"
    from_date="2023-11-01",
    period="P3M",  # dit mag niet icm from_date.. to_date
    max_number_of_tsids_to_retrieve=20,
    max_number_of_values_per_tsid=1000000,
)
out_stats = await analyse_timeseries_statistics(ctx, analyse_timeseries_input)
out_trend = await analyse_timeseries_trend(ctx, analyse_timeseries_input)

# ts_id_in = "37481042"
# find station_no belonging to this ts_id
threshold = wlev_thresholds_dict[station_no]

# for threshold an extra input is required
analyse_timeseries_input = AnalyseTimeseriesThresholdsInput(
    ts_id=ts_id_in,  # belongs to stat_no "1198_WTH_BOV"
    from_date="2023-11-01",
    period="P3M",  # dit mag niet icm from_date.. to_date
    max_number_of_tsids_to_retrieve=20,
    max_number_of_values_per_tsid=1000000,
    threshold=threshold,
)

out_thresholds = await analyse_timeseries_thresholds(ctx, analyse_timeseries_input)

print(out_stats.model_dump())
print(out_trend.model_dump())
print(out_thresholds.model_dump())

{'data': [{'station_name': 'Bovenstrooms', 'station_no': '1198_WTH_BOV', 'station_id': '12516', 'ts_id': '4058042', 'ts_name': 'Jaar.Gem', 'parametertype_id': '11175', 'parametertype_name': 'WNS8487', 'ts_type_id': '14042', 'ts_type_name': 'aggmean'}, {'station_name': 'Bovenstrooms', 'station_no': '1198_WTH_BOV', 'station_id': '12516', 'ts_id': '37481042', 'ts_name': 'Uur.Gem', 'parametertype_id': '11175', 'parametertype_name': 'WNS8487', 'ts_type_id': '14042', 'ts_type_name': 'aggmean'}, {'station_name': 'Bovenstrooms', 'station_no': '1198_WTH_BOV', 'station_id': '12516', 'ts_id': '4056042', 'ts_name': 'Dag.Gem', 'parametertype_id': '11175', 'parametertype_name': 'WNS8487', 'ts_type_id': '14042', 'ts_type_name': 'aggmean'}, {'station_name': 'Bovenstrooms', 'station_no': '1198_WTH_BOV', 'station_id': '12516', 'ts_id': '4055042', 'ts_name': 'ContinueMeting.P', 'parametertype_id': '11175', 'parametertype_name': 'WNS8487', 'ts_type_id': '6042', 'ts_type_name': 'cmd'}, {'station_name': 'Bo

In [4]:
## maak hier nu een plotly fig van

# maak een df van de bovenstaande gebruikte data
from talk_with_wiski.models import GetTimeseriesValuesInput
from talk_with_wiski.tools import get_timeseries_values
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# uitlezen van de data voor zelfde ts_id_in
get_timeseries_input = GetTimeseriesValuesInput(
    ts_id=ts_id_in,  # belongs to stat_no "1198_WTH_BOV"
    from_date="2023-11-01",
    period="P3M",
    max_number_of_tsids_to_retrieve=20,
    max_number_of_values_per_tsid=1000000,
    include_quality_codes=True,
)
out_timeseries_values = await get_timeseries_values(ctx, get_timeseries_input)
# maak hier een dataframe van
data = out_timeseries_values.model_dump()

records = [
    {"timestamp": timestamp, "value": value, "quality_code": quality_code}
    for timestamp, value, quality_code in data["timeseries_values_list"][0]["data"]
]

df = pd.DataFrame(records)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["timestamp"]).reset_index(drop=True)
# Convert timestamp to numeric (seconds since start)
df.loc[:, "time_numeric"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds()
X = df["time_numeric"].values / 86400

slope = out_trend.linear_trend_line.slope
intercept = out_trend.linear_trend_line.intercept

y_fit = [slope * xi + intercept for xi in X]
df["trend"] = y_fit
slope_per_day_cm = slope * 100
df["station_no"] = station_no

fig = px.line(
    df,
    x="timestamp",
    y="value",
    color="station_no",
    title=f"Timeseries h(t) \n Aantal uur boven threshold: {out_thresholds.threshold_values.total_time_above_in_hours}",
)

# threshold = wlev_thresholds_dict[stat_no]
fig.add_hline(
    y=threshold,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Threshold = {out_thresholds.threshold_values.threshold_value}",
    annotation_position="top left",
)
fig.add_trace(
    go.Scatter(
        x=df["timestamp"],
        y=df["trend"],
        line_dash="dash",
        line_color="green",
        # annotation_text=f"Linear trend = {slope_per_day:.2f} cm/dag", annotation_position="top right"
        name="linear trend",
    )
)
# Add annotation for slope/intercept
annot_index = len(df["timestamp"]) // 4
fig.add_annotation(
    x=df["timestamp"].iloc[annot_index],  # place at last timestamp
    y=df["trend"].iloc[annot_index],  # y-coordinate on trend line
    text=f"Slope: {slope_per_day_cm:.2f} cm/dag<br>Intercept: {intercept:.2f} cm",
    # showarrow=True,
    # arrowhead=2,
    # ax=-50,  # offset x of the arrow
    # ay=-50,  # offset y of the arrow
    bgcolor="white",
    bordercolor="black",
    align="right",
)

fig.show()

[{'ts_id': '37481042', 'station_name': 'Bovenstrooms', 'station_latitude': '51.6069011244182', 'station_longitude': '4.99564415135788', 'parametertype_name': 'WNS8487', 'ts_name': 'Uur.Gem', 'ts_unitname': 'meter NAP', 'ts_unitsymbol': 'mNAP', 'station_no': '1198_WTH_BOV', 'station_id': '12516', 'rows': '2209', 'columns': 'Timestamp,Value,Quality Code', 'data': [['2023-11-01T00:00:00.000Z', 5.163, 200], ['2023-11-01T01:00:00.000Z', 5.162, 200], ['2023-11-01T02:00:00.000Z', 5.163, 200], ['2023-11-01T03:00:00.000Z', 5.164, 200], ['2023-11-01T04:00:00.000Z', 5.165, 200], ['2023-11-01T05:00:00.000Z', 5.165, 200], ['2023-11-01T06:00:00.000Z', 5.164, 200], ['2023-11-01T07:00:00.000Z', 5.161, 200], ['2023-11-01T08:00:00.000Z', 5.16, 200], ['2023-11-01T09:00:00.000Z', 5.16, 200], ['2023-11-01T10:00:00.000Z', 5.161, 200], ['2023-11-01T11:00:00.000Z', 5.161, 200], ['2023-11-01T12:00:00.000Z', 5.16, 200], ['2023-11-01T13:00:00.000Z', 5.161, 200], ['2023-11-01T14:00:00.000Z', 5.161, 200], ['2023-1

In [5]:
df

,timestamp,value,quality_code,time_numeric,trend,station_no
0,2023-11-01 00:00:00+00:00,5.163,200,0.0,5.199259,1198_WTH_BOV
1,2023-11-01 01:00:00+00:00,5.162,200,3600.0,5.199246,1198_WTH_BOV
2,2023-11-01 02:00:00+00:00,5.163,200,7200.0,5.199234,1198_WTH_BOV
3,2023-11-01 03:00:00+00:00,5.164,200,10800.0,5.199221,1198_WTH_BOV
4,2023-11-01 04:00:00+00:00,5.165,200,14400.0,5.199209,1198_WTH_BOV
...,...,...,...,...,...,...
2204,2024-01-31 20:00:00+00:00,5.169,200,7934400.0,5.171779,1198_WTH_BOV
2205,2024-01-31 21:00:00+00:00,5.169,200,7938000.0,5.171766,1198_WTH_BOV
2206,2024-01-31 22:00:00+00:00,5.169,200,7941600.0,5.171754,1198_WTH_BOV
2207,2024-01-31 23:00:00+00:00,5.169,200,7945200.0,5.171741,1198_WTH_BOV


In [6]:
# Figure including Quality Code on seconds axis

fig = go.Figure()

fig = px.line(
    df,
    x="timestamp",
    y="value",
    color="station_no",
    title=f"Timeseries h(t) \n Aantal uur boven threshold: {out_thresholds.threshold_values.total_time_above_in_hours}",
)

# threshold = wlev_thresholds_dict[stat_no]
fig.add_hline(
    y=threshold,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Threshold = {out_thresholds.threshold_values.threshold_value}",
    annotation_position="top left",
)
fig.add_trace(
    go.Scatter(
        x=df["timestamp"],
        y=df["trend"],
        yaxis="y1",
        line_dash="dash",
        line_color="green",
        # annotation_text=f"Linear trend = {slope_per_day:.2f} cm/dag", annotation_position="top right"
        name="linear trend",
    )
)
fig.add_trace(
    go.Scatter(
        x=df["timestamp"],
        y=df["quality_code"],
        yaxis="y2",
        line_dash="solid",
        line_color="red",
        # annotation_text=f"Linear trend = {slope_per_day:.2f} cm/dag", annotation_position="top right"
        name="quality code",
    )
)
# Add annotation for slope/intercept
annot_index = len(df["timestamp"]) // 4
fig.add_annotation(
    x=df["timestamp"].iloc[annot_index],  # place at last timestamp
    y=df["trend"].iloc[annot_index],  # y-coordinate on trend line
    text=f"Slope: {slope_per_day_cm:.2f} cm/dag<br>Intercept: {intercept:.2f} cm",
    # showarrow=True,
    # arrowhead=2,
    # ax=-50,  # offset x of the arrow
    # ay=-50,  # offset y of the arrow
    bgcolor="white",
    bordercolor="black",
    align="right",
)

# Layout with two y-axes
fig.update_layout(
    title="Values with Linear Trend and Quality Code",
    xaxis=dict(title="Date"),
    yaxis=dict(
        title="Values (left)",  # Left y-axis label
        side="left",
    ),
    yaxis2=dict(
        title="Quality Code (right)",  # Right y-axis label
        overlaying="y",
        side="right",
    ),
    legend=dict(x=0.1, y=1.1, orientation="h"),
)
# Set figure size
fig.update_layout(
    width=1200,  # width in pixels
    height=600,  # height in pixels
)

fig.show()

In [7]:
from talk_with_wiski.models import GetQualityCodeTableInput
from talk_with_wiski.tools import get_quality_code_table

unique_codes = sorted(df["quality_code"].unique())
print(unique_codes)

# uitlezen van de data voor zelfde ts_id_in
get_qualcode_input = GetQualityCodeTableInput()
qualcodes = await get_quality_code_table(ctx, get_qualcode_input)

# print(qualcodes.data)

# Create df_codes from dictionary
df_codes = pd.DataFrame.from_dict(qualcodes.data, orient="index").reset_index()
df_codes.rename(columns={"index": "key"}, inplace=True)

# Count occurrences in your dataset
counts = df["quality_code"].value_counts().reset_index()
counts.columns = ["key", "count"]

# List of unique codes in your dataset
unique_codes = df["quality_code"].unique()

# Build mapped list
mapped_list = []
for k in unique_codes:
    if k in df_codes["key"].values:
        row = df_codes[df_codes["key"] == k].iloc[0]  # get the row
        mapped_list.append(
            {
                "key": k,
                "code": row["code"],
                "description": row["description"],
                "found": True,
            }
        )
    else:
        mapped_list.append(
            {"key": k, "code": None, "description": None, "found": False}
        )

# Convert to DataFrame
df_mapped = pd.DataFrame(mapped_list)

# Merge with counts
merged_df = pd.merge(df_mapped, counts, on="key", how="left")
merged_df["count"] = merged_df["count"].fillna(0).astype(int)

# print(df_codes)
# print(df_mapped)
# print(counts)
# print(merged_df)

merged_df = merged_df.sort_values(by="key").reset_index(drop=True)

[np.int64(-1), np.int64(150), np.int64(200)]


In [8]:
print("\ndict met alle codes\n")

df_codes


dict met alle codes



,key,code,description
0,40,Betrouwaar,---
1,60,Geschat,---
2,80,HandCorr (oud),handmatig gecorrigeerde data (HandCorr)
3,100,AutoValide (oud),automatisch goedgekeurde data (AutoValide)
4,110,Twijfelachtig,---
5,150,AutoGevuld (oud),automatisch aangevulde data (AutoGevuld)
6,180,Onbetrouwbaar,---
7,190,Gecensureerd,---
8,200,Ongecontroleerd,---
9,254,Afwezig (oud),afgekeurde of ontbrekende data (Afwezig)


In [9]:
print("\nwaarden in dataset\n")

merged_df


waarden in dataset



,key,code,description,found,count
0,-1,None,None,False,26
1,150,AutoGevuld (oud),automatisch aangevulde data (AutoGevuld),True,1
2,200,Ongecontroleerd,---,True,2182
